# Second-round revision: reproduction notebook

This notebook reproduces the second-round analyses from the released data and code. It is light by design: the full runs (equal-work, calibration, timing) take hours and are launched from the scripts in `code/revision2/`.

Run all cells from the repository root.

In [ ]:
import os, sys, json
import pandas as pd
ROOT = os.path.abspath('.')
sys.path.insert(0, os.path.join(ROOT, 'code', 'revision2'))
sys.path.insert(0, os.path.join(ROOT, 'code'))
print('root:', ROOT)

## 1. Paired comparisons (baseline and calibrated objective)

Paired t-test, 95% t-interval and Wilcoxon signed-rank for each comparison. Positive mean difference means the first-named method loses more flows. Bonferroni is over six comparisons per scenario (column `p_bonf_m6`).

In [ ]:
comp = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'revision_comparisons.csv'))
comp

## 2. Calibration validation (seeds 100–104, S2)

Mean PL_flow after repair for each admission bonus mu. mu* = 20 is the choice used in the manuscript.

In [ ]:
val = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'e4a_mu_validation.csv'))
val.groupby('mu')[['PL_flow','energy_sample','energy_all_rejected']].mean().round(2)

## 3. End-to-end timing (S2, 30 seeds, run alone)

Solve and coordinator time are measured separately. Pod time is the sum of the four sequential pod solves. These are single-machine measurements on a two-core sandbox.

In [ ]:
summ = pd.read_csv(os.path.join(ROOT, 'data', 'revision2', 'e23_summary.csv'))
summ

## 4. Software check: release pipeline on S1, seed 42

The release `run_mas` must reproduce the archived MAS-QA result for seed 42 (PL_flow = 26.0, BWL = 32.9926). This takes a few seconds.

In [ ]:
import masqa_lib as L
import r2lib as R
flows = L.gen_traffic(50, 640.0, 42)
cc = R.cc_of(flows)
served = L.run_mas(flows, 42, 640.0, cc)
served = served[0] if isinstance(served, tuple) else served
m = L.metrics(flows, served, 640.0)
print('PL_flow =', m['PL_flow'], ' BWL =', round(m['BWL'], 4))
assert m['PL_flow'] == 26.0

## 5. Energy check: exhaustive enumeration of a small QUBO

The 3-flow, 2-arc instance has 19 binary variables (3 admission, 4 path, 12 slack), so 2^19 = 524,288 states. The script checks that the global minimum is the all-rejected state. It needs a few minutes and about 100 MB of memory. The stored result is in `data/revision2/d_enum.json`.

In [ ]:
print(open(os.path.join(ROOT, 'data', 'revision2', 'd_enum.json')).read())
# To recompute: %run code/revision2/d_enum.py